In [11]:
# brief
# get_mel_spectrogram	Converts audio waveform to Mel spectrogram
# prepare_text	Converts text input to phonemes
# diffusion_schedule	Defines noise schedule for diffusion process
# forward_process	Adds noise to audio data
# denoise_step	Removes noise using a neural network
# Generator	U-Net like structure for generating speech
# Discriminator	Evaluates generated speech quality
# adversarial_loss	Loss function for GAN training
# diffusion_loss	Loss function for noise prediction
# train_step	Performs one training step for both Generator and Discriminator

#LibSpeech dataset

# Text Processing: Handles the conversion of input text into suitable features for the model.​
# arXiv
# +1
# ResearchGate
# +1

# Acoustic Model: Implements the model that maps text features to mel-spectrograms.​

# Diffusion Decoder: Contains the diffusion model enhanced with GAN training for high-fidelity spectrogram generation.​

# Vocoder: Utilizes models like HiFi-GAN to synthesize waveforms from mel-spectrograms.

Tutorials
https://github.com/keonlee9420/DiffGAN-TTS
https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html

In [12]:
# Core Parts of a DiffGAN-TTS-style TTS system:
# Text Encoder – Turns input text into embeddings.

# Acoustic Predictor (e.g. mel-spectrogram generator) – Uses a diffusion model + GAN to predict a mel-spectrogram.

# Vocoder – Converts mel-spectrogram to waveform (e.g., use pre-trained HiFi-GAN or similar).

In [13]:
# Steps to train with dataset
# Load LibriSpeech Dataset (via torchaudio)

# Text Preprocessing (convert transcript to token IDs)

# Audio Preprocessing (convert waveform to mel-spectrogram)

# Batch and Feed to Model

# Train using your train loop

In [ ]:
# Load one .wav file

# Get its transcript

# Convert it to token sequence and mel-spectrogram

# Pass it through your DiffGAN_TTS model

# See if it outputs something reasonable

In [4]:
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import string
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
import torch.nn.utils.rnn as rnn_utils
import pandas as pd
import os

In [9]:
#PreProcessing Funcs
CHAR_VOCAB = {c: i+1 for i, c in enumerate(string.ascii_lowercase + " '")}
CHAR_VOCAB["<pad>"] = 0

def text_to_sequence(text):
    return [CHAR_VOCAB.get(c, 0) for c in text.lower() if c in CHAR_VOCAB]

In [11]:
mel_transform = T.MelSpectrogram(
    sample_rate=16000,
    n_fft=1024,
    hop_length=256,
    n_mels=80
)

def waveform_to_mel(waveform):
    mel = mel_transform(waveform)
    mel_db = torchaudio.transforms.AmplitudeToDB()(mel)
    return mel_db.squeeze(0).transpose(0, 1)  # (Time, Mel)

In [13]:
# --- Load one file ---
# metadata_path = 'LJSpeech-1.1\LJSpeech-1.1\metadata.csv'
metadata_path = r"LJSpeech-1.1\LJSpeech-1.1\metadata.csv"

df = pd.read_csv(metadata_path, sep='|', header=None, names=["file_id", "text", "normalized_text"])

# Example sample
file_id = df.iloc[0]["file_id"]
text = df.iloc[0]["normalized_text"]
wav_path = os.path.join("LJSpeech-1.1\LJSpeech-1.1\wavs", f"{file_id}.wav")

# --- Preprocess ---
waveform, sample_rate = torchaudio.load(wav_path)
text_seq = torch.tensor([text_to_sequence(text)])  # (1, seq_len)
mel = waveform_to_mel(waveform)                    # (time, mel)

print("Text:", text)
print("Input tokens:", text_seq.shape)
print("Mel target shape:", mel.shape)


Text: Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition
Input tokens: torch.Size([1, 149])
Mel target shape: torch.Size([832, 80])


<>:10: SyntaxWarning: invalid escape sequence '\L'
<>:10: SyntaxWarning: invalid escape sequence '\L'
C:\Users\spenc\AppData\Local\Temp\ipykernel_29688\2835279723.py:10: SyntaxWarning: invalid escape sequence '\L'
  wav_path = os.path.join("LJSpeech-1.1\LJSpeech-1.1\wavs", f"{file_id}.wav")


In [ ]:
class LibriSpeechTTS(Dataset):
    def __init__(self, subset):
        self.data = LIBRISPEECH("C:\Users\spenc\OneDrive\Documents\QMUL Documents\ECS7013P\Project Ideas\Assignment 1\LJSpeech-1.1\LJSpeech-1.1\wavs", url=subset, download=True)

    def __getitem__(self, idx):
        waveform, sr, text, *_ = self.data[idx]
        text_seq = torch.tensor(text_to_sequence(text))
        mel = waveform_to_mel(waveform)
        return text_seq, mel

    def __len__(self):
        return len(self.data)

def collate_fn(batch):
    texts, mels = zip(*batch)

    text_lens = torch.tensor([len(t) for t in texts])
    mel_lens = torch.tensor([len(m) for m in mels])

    padded_texts = rnn_utils.pad_sequence(texts, batch_first=True)
    padded_mels = rnn_utils.pad_sequence(mels, batch_first=True)

    return padded_texts, padded_mels, text_lens, mel_lens


In [16]:
# -----------------------
# TEXT ENCODER
# -----------------------
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)  # (B, T_text, E)
        out, _ = self.lstm(x)
        return out  # (B, T_text, H)
    
# -----------------------
# DIFFUSION BLOCK (Simplified)
# -----------------------
class DiffusionBlock(nn.Module):
    def __init__(self, mel_dim, hidden_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(mel_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, mel_dim)
        )

    def forward(self, x, noise_step):
        # Add simple diffusion noise
        noise = torch.randn_like(x) * (1.0 - noise_step / 1000)
        x = x + noise
        x = self.mlp(x)
        return x
    

# -----------------------
# GENERATOR (Mel Generator)
# -----------------------
class MelGenerator(nn.Module):
    def __init__(self, hidden_dim, mel_dim, diffusion_steps=10):
        super().__init__()
        self.diffusion_blocks = nn.ModuleList(
            [DiffusionBlock(mel_dim, hidden_dim) for _ in range(diffusion_steps)]
        )

    def forward(self, x):
        for step, block in enumerate(self.diffusion_blocks):
            x = block(x, step)
        return x
    
# -----------------------
# DISCRIMINATOR (Simplified)
# -----------------------
class Discriminator(nn.Module):
    def __init__(self, mel_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(mel_dim, 128),
            nn.LeakyReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)
    
# -----------------------
# DIFFGAN-TTS WRAPPER
# -----------------------
class DiffGAN_TTS(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, mel_dim):
        super().__init__()
        self.text_encoder = TextEncoder(vocab_size, embed_dim, hidden_dim)
        self.text_to_mel_proj = nn.Linear(hidden_dim, mel_dim)  # <-- ADD THIS LINE
        self.mel_generator = MelGenerator(hidden_dim, mel_dim)
        self.discriminator = Discriminator(mel_dim)

    # def forward(self, text_input):
    #     text_features = self.text_encoder(text_input)
    #     mel_output = self.mel_generator(text_features)
    #     return mel_output
    def forward(self, text_input):
        text_features = self.text_encoder(text_input)   # (B, T, hidden_dim)
        projected = self.text_to_mel_proj(text_features)  # (B, T, mel_dim)
        mel_output = self.mel_generator(projected)
        return mel_output

In [26]:
# Assume text_to_sequence has vocab size ~30
vocab_size = len(CHAR_VOCAB)
embed_dim = 64
hidden_dim = 128
mel_dim = 80

model = DiffGAN_TTS(vocab_size, embed_dim, hidden_dim, mel_dim)
optimizer_g = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer_d = torch.optim.Adam(model.discriminator.parameters(), lr=0.001)

# Forward pass
with torch.no_grad():
    output_mel = model(text_seq)

print("Output mel shape:", output_mel.shape)


Output mel shape: torch.Size([1, 17, 80])


In [27]:
def train_step(generator, discriminator, optimizer_g, optimizer_d, real_mels, text_input, criterion):
    # Generator forward
    fake_mels = generator(text_input)

    # Discriminator loss
    real_score = discriminator(real_mels.detach())
    fake_score = discriminator(fake_mels.detach())
    d_loss = -torch.mean(torch.log(real_score) + torch.log(1. - fake_score))

    optimizer_d.zero_grad()
    d_loss.backward()
    optimizer_d.step()

    # Generator loss (minimize discriminator feedback + reconstruction)
    fake_score = discriminator(fake_mels)
    g_loss = -torch.mean(torch.log(fake_score)) + criterion(fake_mels, real_mels)

    optimizer_g.zero_grad()
    g_loss.backward()
    optimizer_g.step()

    return d_loss.item(), g_loss.item()


In [28]:
text = ['the first idea is']

In [29]:
text_seq = torch.tensor([text_to_sequence(t) for t in text])

In [30]:
train_step(model, 
           Discriminator(mel_dim), 
           optimizer_g, 
           optimizer_d, 
           text_seq, 
           torch.nn.MSELoss())

TypeError: train_step() missing 1 required positional argument: 'criterion'

In [ ]:
def mel_to_waveform(mel_spec, sample_rate=22050, n_fft=1024, hop_length=256):
    # Convert from mel-spectrogram to linear spectrogram
    mel_spec = mel_spec.transpose(1, 2)  # (B, mel_dim, T)
    mel_basis = torchaudio.functional.create_fb_matrix(n_fft // 2 + 1, sample_rate)
    mel_spec_linear = torch.matmul(torch.pinverse(mel_basis), mel_spec)

    # Invert with Griffin-Lim
    griffin_lim = torchaudio.transforms.GriffinLim(
        n_fft=n_fft,
        hop_length=hop_length,
        n_iter=32
    )
    waveform = griffin_lim(mel_spec_linear)
    return waveform